In [41]:
# Imports

import os
import sys
import json
import pandas as pd
import numpy as np
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

In [42]:
# ── Set Working Directory to Project Root ───────────────────────────────────────
def find_project_root(folder_name="Project-Omnilection"):
    try:
        current_dir = os.path.abspath(os.path.dirname(__file__))
    except NameError:
        # Running in Jupyter
        current_dir = os.getcwd()
    while True:
        if os.path.basename(current_dir) == folder_name:
            return current_dir
        parent = os.path.dirname(current_dir)
        if parent == current_dir:
            break
        current_dir = parent
    raise FileNotFoundError(f"Could not find {folder_name}")

def set_directory():
    target_folder = find_project_root()
    os.chdir(target_folder)
    if target_folder not in sys.path:
        sys.path.insert(0, target_folder)

set_directory()

In [43]:
# %%
# ── 1. Data Ingestion & Formatting ───────────────────────────────────────────
import json
import pandas as pd

# Point this to your target run directory
RUN_DIR = "test_run_2026_06_18_045952" # CHANGE THIS TO YOUR LATEST RUN FOLDER
HERITAGE_FILE = f"{RUN_DIR}/heritage_data.json"

with open(HERITAGE_FILE, "r") as f:
    heritage_db = json.load(f).get("prompt_chains", {})

records = []
for chain_id, data in heritage_db.items():
    meta = data.get("metadata", {})
    
    # 🚨 FIX: Safely handle JSON nulls
    raw_fitness = data.get("fitness")
    fitness = float(raw_fitness) if raw_fitness is not None else 0.0
    
    parents = data.get("parents", [])
    
    # Extract telemetry if available
    accuracy = meta.get("accuracy", 0.0)
    time_taken = meta.get("time_taken", 0.0)
    token_usage = meta.get("token_usage", 0.0)
    
    mutated = meta.get("mutated", False)
    recombination_mode = meta.get("recombination_mode", "unknown")
    
    lineage_score = meta.get("lineage_score", fitness * 0.8) 
    
    # Safely handle the generation list
    generations = data.get("generation", [])
    if not isinstance(generations, list):
        generations = [generations]
        
    for gen in generations:
        records.append({
            "chain_id": chain_id,
            "generation": gen,
            "fitness": fitness,
            "lineage_score": lineage_score,
            "parents": parents,
            "num_parents": len(parents),
            "mutated": mutated,
            "recombination_mode": recombination_mode,
            "accuracy": accuracy,
            "time_taken": time_taken,
            "token_usage": token_usage
        })

df = pd.DataFrame(records)
print(f"Loaded {len(df)} generational records from {len(heritage_db)} unique chains.")

Loaded 4582 generational records from 2820 unique chains.


In [44]:
# ## 2. Global Fitness & Lineage Dynamics
# Average and top fitness score by generation.

# %%
gen_stats = df.groupby('generation').agg(
    max_fitness=('fitness', 'max'),
    avg_fitness=('fitness', 'mean'),
    avg_lineage=('lineage_score', 'mean'),
    pop_size=('chain_id', 'count')
).reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=gen_stats['generation'], y=gen_stats['max_fitness'], mode='lines+markers', name='Max Fitness', line=dict(color='green', width=3)))
fig.add_trace(go.Scatter(x=gen_stats['generation'], y=gen_stats['avg_fitness'], mode='lines+markers', name='Avg Fitness', line=dict(color='blue', width=2)))
fig.add_trace(go.Scatter(x=gen_stats['generation'], y=gen_stats['avg_lineage'], mode='lines+markers', name='Avg Lineage', line=dict(color='purple', dash='dash')))

fig.update_layout(title="Fitness & Lineage Trajectories over Generations", xaxis_title="Generation", yaxis_title="Score")
fig.show()

In [45]:
# ## 3. Lineage Score Distribution (Interactive Slider)
# Watch how the population shifts from chaotic (wide distribution) to stable (normal distribution) over time.

# %%
# Ensure generation is sorted for the animation frame
df = df.sort_values('generation')
fig = px.histogram(
    df, x="lineage_score", animation_frame="generation", 
    range_x=[0, 1], nbins=20, color_discrete_sequence=['indigo'],
    title="Lineage Score Distribution Shift per Generation"
)
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 800
fig.show()

In [46]:
# ## 4. Fitness Function Telemetry (Fine-Tuning Metrics)
# Analyzes Average Accuracy, Time, and Token usage over time.

# %%
telemetry_stats = df.groupby('generation')[['accuracy', 'time_taken', 'token_usage']].mean().reset_index()

from plotly.subplots import make_subplots
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=("Average Accuracy (%)", "System Load: Time vs Tokens"),
                    specs=[[{"secondary_y": False}], [{"secondary_y": True}]])

# Plot Accuracy (Top Chart)
fig.add_trace(go.Scatter(x=telemetry_stats['generation'], y=telemetry_stats['accuracy']*100, 
                         name="Avg Accuracy", line=dict(color='green', width=3)), row=1, col=1)

# Plot Time and Tokens (Bottom Chart)
fig.add_trace(go.Scatter(x=telemetry_stats['generation'], y=telemetry_stats['time_taken'], 
                         name="Avg Time (s)", line=dict(color='blue', dash='dash')), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=telemetry_stats['generation'], y=telemetry_stats['token_usage'], 
                         name="Avg Tokens", line=dict(color='orange', dash='dot')), row=2, col=1, secondary_y=True)

fig.update_layout(title="Evolutionary Telemetry by Generation", height=700, template="plotly_white")
fig.update_yaxes(title_text="Accuracy Score", row=1, col=1)
fig.update_yaxes(title_text="Execution Time (s)", secondary_y=False, row=2, col=1)
fig.update_yaxes(title_text="Tokens Used", secondary_y=True, row=2, col=1)
fig.update_xaxes(title_text="Generation", row=2, col=1)
fig.show()

In [47]:
# ## 5. Recombination Mode Efficacy
# Evaluates Average Success Rate (Offspring Fitness > Avg Parent Fitness).

# %%
# Rebuild parent relationships to calculate success
recombination_data = []
for idx, row in df[df['num_parents'] == 2].iterrows():
    p1, p2 = row['parents']
    p1_fit = heritage_db.get(p1, {}).get("fitness", 0)
    p2_fit = heritage_db.get(p2, {}).get("fitness", 0)
    
    avg_parent_fit = (p1_fit + p2_fit) / 2
    success = 1 if row['fitness'] > avg_parent_fit else 0
    
    recombination_data.append({
        "generation": row['generation'],
        "mode": row['recombination_mode'],
        "success": success,
        "fitness_gain": row['fitness'] - avg_parent_fit
    })

if recombination_data:
    rec_df = pd.DataFrame(recombination_data)
    rec_stats = rec_df.groupby('mode').agg(
        success_rate=('success', 'mean'),
        avg_fitness_gain=('fitness_gain', 'mean'),
        count=('success', 'count')
    ).reset_index()
    
    fig = px.bar(rec_stats, x='mode', y='success_rate', color='avg_fitness_gain', 
                 title="Recombination Success Rate by Pairing Mode",
                 labels={'success_rate': 'Success Rate (Offspring > Parents)', 'mode': 'Pairing Mode'})
    fig.show()
else:
    print("No valid 2-parent recombination data found (yet).")

In [48]:
# ## 6. Semantic K-Clustering Sweep (Diversity Check)
# How diverse is the gene pool? We sweep K=2 through K=10. A high Silhouette Score at a high K means healthy diversity.

# %%
# To avoid running Ollama in the notebook, we use TF-IDF on the string representations of the chains.
# In a true setup, you could load the Ollama embeddings here.

import json
with open(f"{RUN_DIR}/id_to_promptchain.json", "r") as f:
    id_to_chain = json.load(f)

clustering_results = []
vectorizer = TfidfVectorizer(max_features=100)

for gen in sorted(df['generation'].unique()):
    gen_chain_ids = df[df['generation'] == gen]['chain_id'].tolist()
    gen_texts = [str(id_to_chain.get(cid, "")) for cid in gen_chain_ids]
    
    if len(gen_texts) < 10:
        continue # Not enough data to cluster
        
    X = vectorizer.fit_transform(gen_texts)
    
    for k in range(2, min(10, len(gen_texts))):
        kmeans = KMeans(n_clusters=k, random_state=42).fit(X)
        score = silhouette_score(X, kmeans.labels_)
        clustering_results.append({"generation": gen, "k": k, "silhouette": score})

if clustering_results:
    clust_df = pd.DataFrame(clustering_results)
    fig = px.line(clust_df, x="generation", y="silhouette", color="k", 
                  title="Semantic Diversity: Silhouette Score by K-Clusters over Generations")
    fig.show()
else:
    print("Not enough generations or population size to run K-Means.")

In [53]:
# ## 7. Model Topology Evolution Over Time
# Tracks the relative distribution (percentage) of LLM models used across all surviving prompt chains.
# Visualizes how the algorithm organically phases out inefficient models in favor of optimal architectures.

# %%
import os
import json
import plotly.express as px
import pandas as pd
from collections import defaultdict

# 1. Load the ID mapping to get the actual prompt chains
id_to_chain_path = os.path.join(RUN_DIR, "id_to_promptchain.json")
with open(id_to_chain_path, "r") as f:
    id_to_chain = json.load(f)

# 2. Tally the models used in every generation
generation_model_counts = defaultdict(lambda: defaultdict(int))
total_model_usage = defaultdict(int) # Track global usage for legend sorting

for chain_id, data in heritage_db.items():
    chain = id_to_chain.get(chain_id, [])
    
    generations = data.get("generation", [])
    if not isinstance(generations, list):
        generations = [generations]
        
    models_in_chain = []
    for step in chain:
        if isinstance(step, (list, tuple)) and len(step) > 0:
            models_in_chain.append(step[0]) 
            
    for gen in generations:
        for model in models_in_chain:
            generation_model_counts[gen][model] += 1
            total_model_usage[model] += 1

# Order models by highest overall usage (Descending)
ordered_models = sorted(total_model_usage.keys(), key=lambda m: total_model_usage[m], reverse=True)

# 3. Convert to DataFrame and Normalize into Percentages
plot_data = []
for gen, model_counts in generation_model_counts.items():
    total_in_gen = sum(model_counts.values())
    
    for model in ordered_models:
        count = model_counts.get(model, 0)
        percentage = (count / total_in_gen * 100) if total_in_gen > 0 else 0
        
        plot_data.append({
            "Generation": gen, 
            "Model": model, 
            "Percentage": percentage,
            "Raw Count": count
        })

model_df = pd.DataFrame(plot_data)

if not model_df.empty:
    model_df = model_df.sort_values("Generation")
    
    # 4. Plot a Line Chart
    fig = px.line(
        model_df, 
        x="Generation", 
        y="Percentage", 
        color="Model",
        category_orders={"Model": ordered_models}, # Enforce descending legend order
        title="Evolutionary Shift in LLM Model Usage (Normalized Percentage)",
        labels={"Percentage": "Population Share (%)", "Generation": "Generation"},
        color_discrete_sequence=px.colors.qualitative.Bold,
        hover_data={"Raw Count": True} # Show the exact number on hover
    )
    
    fig.update_layout(
        xaxis=dict(showgrid=True), # Removed dtick=1 for auto-scaling
        yaxis=dict(showgrid=True, range=[0, 105]), # Cap Y-axis at 100%
        hovermode="x unified",
        legend_title_text='LLM Models (Click to toggle)',
        plot_bgcolor='rgba(245,245,250,1)',
        height=600
    )
    
    # Make the lines slightly thicker for readability
    fig.update_traces(line=dict(width=3))
    
    fig.show()
else:
    print("No model data found to plot.")

In [ ]:
# ## 8. The Full Genetic Tree (Directed Acyclic Graph)
# Traces the lineage of the absolute best prompt chain from the ACTIVE population back to Generation 0. 
# Left-to-Right layout by Generation, with arrows showing genetic flow.

# %%
import os
import json
import networkx as nx
import plotly.graph_objects as go
from Genetic_algorithm_processes.Data.lineage_scoring import LineageScorer

# 1. Load the ID Mapping
id_to_chain_path = os.path.join(RUN_DIR, "id_to_promptchain.json")
with open(id_to_chain_path, "r") as f:
    id_to_chain = json.load(f)

# 2. Dynamically calculate Lineage Scores
print("Calculating dynamic lineage scores...")
scorer = LineageScorer(
    heritage_data=heritage_db,
    id_to_chain=id_to_chain,
    use_embeddings=False, 
    verbose=False
)
lineage_scores = scorer.compute_all_lineage_scores()

# 🚨 FIX: 3. Find the ultimate champion chain from the ACTIVE POPULATION
pop_file_path = os.path.join(RUN_DIR, "population_data.json")
with open(pop_file_path, "r") as f:
    active_population = json.load(f).get("population", [])

if not active_population:
    raise ValueError("No active population found in population_data.json")

# Population record format: [chain_id, chain_tuples, fitness, metadata]
best_chain_record = max(active_population, key=lambda x: (x[2] if x[2] is not None else 0.0))
best_chain_id = best_chain_record[0]
best_fitness = best_chain_record[2]

print(f"Ultimate Champion (Active Pop): {best_chain_id} (Fitness: {best_fitness:.4f})")

# 4. Build the Directed Acyclic Graph (DAG)
G = nx.DiGraph()

def trace_lineage(chain_id, depth=0):
    if depth > 50: return 
    if chain_id not in heritage_db: return
    
    record = heritage_db[chain_id]
    
    # Extract data
    gen = min(record.get("generation", [0])) 
    fit = record.get("fitness")
    fit = float(fit) if fit is not None else 0.0
    lin = lineage_scores.get(chain_id, 0.0)
    
    G.add_node(chain_id, fitness=fit, lineage=lin, gen=gen)
    
    for p in record.get("parents", []):
        # Directed Edge: Parent -> Child
        G.add_edge(p, chain_id)
        trace_lineage(p, depth + 1)

trace_lineage(best_chain_id)

if len(G.nodes) > 0:
    # Manually assign X (Generation) and Y (Spacing) coordinates
    pos = {}
    
    # Group nodes by generation to space them vertically
    gen_groups = {}
    for n, d in G.nodes(data=True):
        g = d['gen']
        if g not in gen_groups:
            gen_groups[g] = []
        gen_groups[g].append(n)
        
    for g, nodes in gen_groups.items():
        # Sort nodes in each generation by fitness for cleaner layout
        nodes.sort(key=lambda x: G.nodes[x]['fitness'], reverse=True)
        # Space them evenly on the Y axis
        y_spacing = 1.0 / (len(nodes) + 1)
        for i, n in enumerate(nodes):
            pos[n] = (g, 1.0 - (y_spacing * (i + 1)))

    node_x = [pos[n][0] for n in G.nodes()]
    node_y = [pos[n][1] for n in G.nodes()]

    fig = go.Figure()
    
    # Draw Lines WITH ARROWS using Annotations
    annotations = []
    for edge in G.edges():
        x0, y0 = pos[edge[0]] # Parent
        x1, y1 = pos[edge[1]] # Child
        
        # Add an annotation to create the arrow
        annotations.append(
            dict(
                ax=x0, ay=y0, axref='x', ayref='y',
                x=x1, y=y1, xref='x', yref='y',
                showarrow=True,
                arrowhead=2,
                arrowsize=1.5,
                arrowwidth=1.5,
                arrowcolor='#888',
                opacity=0.7
            )
        )
    
    # Draw Nodes
    fitness_vals = [G.nodes[n]['fitness'] for n in G.nodes()]
    hover_texts = [
        f"<b>Chain ID:</b> {n[:12]}...<br>"
        f"<b>Born in Gen:</b> {G.nodes[n]['gen']}<br>"
        f"<b>Fitness:</b> {G.nodes[n]['fitness']:.4f}<br>"
        for n in G.nodes()
    ]
    
    fig.add_trace(go.Scatter(
        x=node_x, y=node_y, mode='markers',
        marker=dict(
            showscale=True, colorscale='Plasma', size=18, 
            color=fitness_vals, colorbar=dict(title="Fitness Score"),
            line=dict(width=2, color='white')
        ),
        text=hover_texts, hoverinfo='text'
    ))
    
    fig.update_layout(
        title=f"Ancestry Tree of Champion {best_chain_id[:8]}", 
        showlegend=False, 
        annotations=annotations, # Inject the arrows
        xaxis=dict(
            title="Generation (Absolute)", 
            showgrid=True, 
            zeroline=False, 
            tickmode='linear',
            dtick=1 # Force ticks at every integer generation
        ), 
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        plot_bgcolor='rgba(245,245,250,1)',
        height=800
    )
    fig.show()
else:
    print("Could not build genetic tree.")

Calculating dynamic lineage scores...
Ultimate Champion (Active Pop): chain_2c47c6b5a32c (Fitness: 0.8605)
